# Augmented Evaluation: Whisper-small

Evaluates Whisper-small on concatenated, noisy, and perturbed code-switching sets.

See the repository README for setup instructions, datasets, results, and limitations.


In [ ]:
# Google Drive ASR Evaluation with Augmented Data
# Processes audio files and CSV from Google Drive

# Install required packages
!pip install datasets jiwer transformers accelerate torch torchaudio opencc-python-reimplemented pandas librosa soundfile --quiet

import torch, jiwer, re, gc, opencc, os, zipfile, librosa, soundfile as sf
import numpy as np
import pandas as pd
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm
from google.colab import drive

In [ ]:
# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("Google Drive mounted")

In [ ]:
# =========================
# GPU Setup for Google Colab (GPU ONLY)
# =========================
print("🔧 Checking for GPU availability...")

if not torch.cuda.is_available():
    print("❌ ERROR: No GPU detected!")
    print("Please enable GPU in Google Colab:")
    raise RuntimeError("GPU is required but not available. Please enable GPU in Colab settings.")

device = torch.device("cuda")
print(f"GPU detected: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")

# Clear any existing GPU cache
torch.cuda.empty_cache()

# =========================
# Load Whisper Model (GPU Optimized)
# =========================
print("\n Loading Whisper-small model...")
model_name = "openai/whisper-small"

# Load with GPU optimizations (GPU REQUIRED)
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # Always use float16 for GPU
    device_map="auto"
)

# Tokenizer setup
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

model.eval()
model = model.to(device)
print(f"Model loaded on GPU")

# Monitor GPU memory
print(f"GPU memory after model loading: {torch.cuda.memory_allocated(0) // 1024**2} MB")

# =========================
# Load Data from Google Drive
# =========================
print("\n Loading data from Google Drive...")

# Set Google Drive folder path
DRIVE_FOLDER = "/content/drive/MyDrive/augmented_data"

# Load metadata.csv and inspect its content
metadata_path = os.path.join(DRIVE_FOLDER, "metadata.csv")
if os.path.exists(metadata_path):
    metadata_df = pd.read_csv(metadata_path)

    # Correct the file paths for Linux environment
    metadata_df['file_path'] = metadata_df['file_path'].str.replace('\\', '/', regex=False)
    # ---------------------------------------------------------

    print(f"  Loaded metadata: {len(metadata_df)} records")
    print(f"   Columns: {list(metadata_df.columns)}")

    # Debug: Show unique values in augmentation_type column
    print(f"\n Debug: Checking augmentation_type values...")
    if 'augmentation_type' in metadata_df.columns:
        unique_aug_types = metadata_df['augmentation_type'].unique()
        print(f"   Unique augmentation_type values: {unique_aug_types}")
        print(f"   Value counts:")
        for aug_type, count in metadata_df['augmentation_type'].value_counts().items():
            print(f"     '{aug_type}': {count} samples")
    else:
        print(f"   ❌ 'augmentation_type' column not found!")
        print(f"   Available columns: {list(metadata_df.columns)}")

    # Show first few rows for inspection
    print(f"\n First 3 rows of metadata:")
    print(metadata_df.head(3))

else:
    raise FileNotFoundError(f"metadata.csv not found at {metadata_path}")

# Extract ZIP files
zip_files = ["test_concat.zip", "test_noisy.zip", "test_perturbed.zip"]
audio_base_path = "/content/audio_data"
os.makedirs(audio_base_path, exist_ok=True)

print("\n Extracting ZIP files...")
for zip_file in zip_files:
    zip_path = os.path.join(DRIVE_FOLDER, zip_file)
    if os.path.exists(zip_path):
        extract_path = os.path.join(audio_base_path, zip_file.replace('.zip', ''))
        os.makedirs(extract_path, exist_ok=True)

        print(f"   Extracting {zip_file}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(f"   ✅ {zip_file} extracted to {extract_path}")
    else:
        print(f"   ⚠️ {zip_file} not found")

# =========================
# MER Calculation Function
# =========================
# Initialize OpenCC converters for Chinese conversion needs
converter_s2t = opencc.OpenCC('s2t')  # Simplified to Traditional
converter_t2s = opencc.OpenCC('t2s')  # Traditional to Simplified

def compute_mer(ref, hyp, dataset_name="Custom", debug=False):
    """
    Calculate Mixed Error Rate for code-switching text
    Reference is already Traditional Chinese, only convert hypothesis
    """
    try:
        # Clean and normalize text
        ref = ref.strip()
        hyp = hyp.strip()

        # Since all references are Traditional Chinese, only convert hypothesis
        ref = ref  # Keep original reference (already Traditional Chinese)
        hyp = converter_s2t.convert(hyp)  # Convert Whisper output to Traditional Chinese

        if debug:
            print(f"      Dataset: {dataset_name} (Traditional Chinese)")
            print(f"      Original Hyp: '{hyp}'")
            print(f"      Ref (Traditional): '{ref}'")
            print(f"      Hyp (Traditional): '{hyp}'")

        # Separate Chinese and English
        chinese_ref = re.sub(r"[A-Za-z0-9\s]", "", ref).strip()
        chinese_hyp = re.sub(r"[A-Za-z0-9\s]", "", hyp).strip()
        english_ref = re.sub(r"[^A-Za-z\s]", "", ref).strip()
        english_hyp = re.sub(r"[^A-Za-z\s]", "", hyp).strip()

        # Remove extra spaces
        english_ref = ' '.join(english_ref.split())
        english_hyp = ' '.join(english_hyp.split())

        if debug:
            print(f"      Chinese Ref (Traditional): '{chinese_ref}' (len={len(chinese_ref)})")
            print(f"      Chinese Hyp (Traditional): '{chinese_hyp}' (len={len(chinese_hyp)})")
            print(f"      English Ref: '{english_ref}' (words={len(english_ref.split()) if english_ref else 0})")
            print(f"      English Hyp: '{english_hyp}' (words={len(english_hyp.split()) if english_hyp else 0})")

        # Chinese character-level error calculation
        if chinese_ref or chinese_hyp:
            if not chinese_ref and chinese_hyp:
                ins_c, del_c, sub_c, n_c = len(chinese_hyp), 0, 0, len(chinese_hyp)
            elif chinese_ref and not chinese_hyp:
                ins_c, del_c, sub_c, n_c = 0, len(chinese_ref), 0, len(chinese_ref)
            else:
                cer_measures = jiwer.process_characters([chinese_ref], [chinese_hyp])
                ins_c = cer_measures.insertions
                del_c = cer_measures.deletions
                sub_c = cer_measures.substitutions
                n_c = len(chinese_ref)
        else:
            ins_c, del_c, sub_c, n_c = 0, 0, 0, 0

        # English word-level error calculation
        if english_ref or english_hyp:
            if not english_ref and english_hyp:
                ins_w, del_w, sub_w, n_w = len(english_hyp.split()), 0, 0, len(english_hyp.split())
            elif english_ref and not english_hyp:
                ins_w, del_w, sub_w, n_w = 0, len(english_ref.split()), 0, len(english_ref.split())
            else:
                wer_measures = jiwer.process_words([english_ref], [english_hyp])
                ins_w = wer_measures.insertions
                del_w = wer_measures.deletions
                sub_w = wer_measures.substitutions
                n_w = len(english_ref.split())
        else:
            ins_w, del_w, sub_w, n_w = 0, 0, 0, 0

        if debug:
            print(f"      Chinese errors: I={ins_c}, D={del_c}, S={sub_c}, N={n_c}")
            print(f"      English errors: I={ins_w}, D={del_w}, S={sub_w}, N={n_w}")

        # Combine errors
        total_ins = ins_c + ins_w
        total_del = del_c + del_w
        total_sub = sub_c + sub_w
        total_ref = n_c + n_w

        if total_ref == 0:
            return 100.0 if (chinese_hyp or english_hyp) else 0.0

        mer = (total_ins + total_del + total_sub) / total_ref * 100

        if debug:
            print(f"      Final: Total_errors={total_ins+total_del+total_sub}, Total_ref={total_ref}, MER={mer:.2f}%")

        return mer

    except Exception as e:
        print(f"    ❌ MER calculation error: {e}")
        return 100.0

# =========================
# Audio Processing Function
# =========================
def process_audio_file(audio_path, model, processor, device):
    try:
        # Load audio file
        audio_array, sample_rate = librosa.load(audio_path, sr=16000, mono=True)

        # Ensure float32 format
        audio_array = audio_array.astype(np.float32)

        # Pad/trim to 30 seconds (Whisper requirement)
        target_length = 16000 * 30  # 30 seconds at 16kHz

        if len(audio_array) < target_length:
            padded_audio = np.zeros(target_length, dtype=np.float32)
            padded_audio[:len(audio_array)] = audio_array
            audio_array = padded_audio
        else:
            audio_array = audio_array[:target_length]

        # Extract features
        input_features = processor.feature_extractor(
            audio_array,
            sampling_rate=16000,
            return_tensors="pt"
        )

        # Convert to float16 and move to device
        input_features = input_features.input_features.to(device, dtype=torch.float16)

        # Generate transcription
        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                max_new_tokens=256,
                num_beams=1,
                do_sample=False,
                use_cache=True
            )

        # Decode
        pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        # Clean up GPU memory
        del input_features, predicted_ids
        torch.cuda.empty_cache()

        return pred_text

    except Exception as e:
        print(f"Error processing audio file {audio_path}: {e}")
        return ""

# =========================
# Evaluate Each Test Set Separately
# =========================
def evaluate_test_set(metadata_df, audio_base_path, test_set_name, debug_samples=5):
    """Evaluate a specific test set (concat, noisy, or perturbed)"""
    # Filter metadata for this specific test set
    test_df = metadata_df[metadata_df['augmentation_type'] == test_set_name].copy()

    if len(test_df) == 0:
        print(f" No samples found for {test_set_name}")
        return []

    print(f"\n Processing {test_set_name} dataset ({len(test_df)} samples)")
    print(f"   Debug mode for first {debug_samples} samples")

    results = []
    processed_count = 0

    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Processing {test_set_name}"):
        try:
            file_path = row['file_path']
            transcript = row['transcript']
            aug_type = row['augmentation_type']
            extra_info = row.get('extra_info', '')

            # Construct full audio file path directly from the base directory
            if file_path.startswith("test_concat/"):
              audio_path = os.path.join(audio_base_path, "test_concat", "test_concat", os.path.basename(file_path))
            elif file_path.startswith("test_noisy/"):
              audio_path = os.path.join(audio_base_path, "test_noisy", "test_noisy", os.path.basename(file_path))
            elif file_path.startswith("test_perturbed/"):
              audio_path = os.path.join(audio_base_path, "test_perturbed", "test_perturbed", os.path.basename(file_path))
            else:
              audio_path = os.path.join(audio_base_path, file_path)


            # Check if audio file exists and process it
            if not os.path.exists(audio_path):
                print(f"    ⚠️ Audio file not found at expected path: {audio_path}")
                continue

            show_debug = processed_count < debug_samples
            if show_debug:
                print(f"\n  📝 Sample {processed_count + 1} ({test_set_name}) - DEBUG MODE")
                print(f"      File: {audio_path}")
                print(f"      Extra info: {extra_info}")

            # Process audio
            pred_text = process_audio_file(audio_path, model, processor, device)

            if pred_text:
                # Calculate MER
                mer = compute_mer(transcript, pred_text, f"{test_set_name}", debug=show_debug)

                if mer >= 0:
                    result = {
                        'file_path': file_path,
                        'augmentation_type': aug_type,
                        'extra_info': extra_info,
                        'transcript': transcript,
                        'hypothesis': pred_text,
                        'mer': mer
                    }
                    results.append(result)
                    processed_count += 1

                    if show_debug:
                        print(f"    📊 MER: {mer:.2f}%")
                        print(f"    📚 Reference: '{transcript}'")
                        print(f"    🎯 Hypothesis: '{pred_text}'")

                        gpu_mem = torch.cuda.memory_allocated(0) // 1024**2
                        print(f"    🎮 GPU Memory: {gpu_mem} MB")
                        print("-" * 50)
                else:
                    print(f"     Invalid MER: {mer:.2f}%")
            else:
                print(f"    ❌ Failed to transcribe: {audio_path}")

        except Exception as e:
            print(f"    ❌ Error processing sample {idx}: {e}")
            continue

        # Memory cleanup every 10 samples
        if (processed_count) % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

        # Progress report every 50 samples
        if (processed_count) % 50 == 0:
            current_avg = sum(r['mer'] for r in results) / len(results) if results else 0
            print(f"  📈 Progress: {len(results)} processed, avg MER: {current_avg:.2f}%")

    print(f"✅ {test_set_name}: {len(results)} samples processed successfully")
    return results

# =========================
# Main Evaluation - Sequential Test Sets
# =========================
print("\n Starting sequential evaluation of three test sets...")
print("="*60)

# Define the test sets to evaluate
test_sets = ["concat", "noisy", "perturbed"]
all_results = {}

# Process each test set sequentially
for test_set in test_sets:
    print(f"\n{'='*20} {test_set.upper()} {'='*20}")

    # Check if this test set exists in metadata
    test_samples = metadata_df[metadata_df['augmentation_type'] == test_set]
    if len(test_samples) == 0:
        print(f" No samples found for {test_set}")
        all_results[test_set] = []
        continue

    print(f"Found {len(test_samples)} samples for {test_set}")

    # Evaluate this test set
    results = evaluate_test_set(metadata_df, audio_base_path, test_set, debug_samples=5)
    all_results[test_set] = results

    # Show immediate results for this test set
    if len(results) > 0:
        test_mers = [r['mer'] for r in results]
        avg_mer = sum(test_mers) / len(test_mers)
        min_mer = min(test_mers)
        max_mer = max(test_mers)

        print(f"\n📊 {test_set} Results:")
        print(f"   Average MER: {avg_mer:.2f}%")
        print(f"   Min MER: {min_mer:.2f}%")
        print(f"   Max MER: {max_mer:.2f}%")
        print(f"   Samples processed: {len(results)}")

        # Save individual test set results
        test_df = pd.DataFrame(results)
        test_output_path = f"/content/{test_set}_results.csv"
        test_df.to_csv(test_output_path, index=False)
        print(f"   Results saved to: {test_output_path}")

    # Memory cleanup between test sets
    gc.collect()
    torch.cuda.empty_cache()
    print(f"✅ {test_set} evaluation completed")

# Combine all results for overall analysis
all_combined_results = []
for test_set, results in all_results.items():
    all_combined_results.extend(results)

# =========================
# Final Comprehensive Results Analysis
# =========================
print("\n" + "="*60)
print("🏆 COMPREHENSIVE RESULTS SUMMARY")
print("="*60)

if len(all_combined_results) > 0:
    # Create comprehensive DataFrame
    comprehensive_df = pd.DataFrame(all_combined_results)

    # Overall statistics across all test sets
    overall_avg = comprehensive_df['mer'].mean()
    overall_min = comprehensive_df['mer'].min()
    overall_max = comprehensive_df['mer'].max()

    print(f"📈 Overall Statistics (All Test Sets):")
    print(f"   Average MER: {overall_avg:.2f}%")
    print(f"   Min MER: {overall_min:.2f}%")
    print(f"   Max MER: {overall_max:.2f}%")
    print(f"   Total samples: {len(all_combined_results)}")

    # Individual test set comparison
    print(f"\n📊 Test Set Comparison:")
    print("-" * 50)
    for test_set in test_sets:
        if test_set in all_results and len(all_results[test_set]) > 0:
            test_mers = [r['mer'] for r in all_results[test_set]]
            avg_mer = sum(test_mers) / len(test_mers)
            std_mer = np.std(test_mers)
            print(f"   {test_set:15}: {avg_mer:6.2f}% ± {std_mer:5.2f}% (n={len(test_mers)})")
        else:
            print(f"   {test_set:15}: No data")

    # Statistical significance test
    test_sets_with_data = [ts for ts in test_sets if ts in all_results and len(all_results[ts]) > 0]
    if len(test_sets_with_data) > 1:
        print(f"\n🔬 Statistical Analysis:")
        for i, ts1 in enumerate(test_sets_with_data):
            for ts2 in test_sets_with_data[i+1:]:
                mers1 = [r['mer'] for r in all_results[ts1]]
                mers2 = [r['mer'] for r in all_results[ts2]]
                avg_diff = np.mean(mers1) - np.mean(mers2)
                print(f"   {ts1} vs {ts2}: Diff = {avg_diff:+.2f}%")

    # Save comprehensive results
    comprehensive_output_path = "/content/comprehensive_results.csv"
    comprehensive_df.to_csv(comprehensive_output_path, index=False)
    print(f"\n💾 Comprehensive results saved to: {comprehensive_output_path}")

    # Show best and worst performing samples
    print(f"\n🎯 Performance Highlights:")
    best_sample = comprehensive_df.loc[comprehensive_df['mer'].idxmin()]
    worst_sample = comprehensive_df.loc[comprehensive_df['mer'].idxmax()]

    print(f"   🏆 Best performance (MER: {best_sample['mer']:.2f}%):")
    print(f"      File: {os.path.basename(best_sample['file_path'])}")
    print(f"      Test set: {best_sample['augmentation_type']}")
    print(f"      Transcript: {best_sample['transcript'][:60]}...")

    print(f"   📉 Worst performance (MER: {worst_sample['mer']:.2f}%):")
    print(f"      File: {os.path.basename(worst_sample['file_path'])}")
    print(f"      Test set: {worst_sample['augmentation_type']}")
    print(f"      Transcript: {worst_sample['transcript'][:60]}...")

    # Performance distribution
    print(f"\n📊 Performance Distribution:")
    mer_ranges = [(0, 20), (20, 40), (40, 60), (60, 80), (80, 100), (100, float('inf'))]
    for low, high in mer_ranges:
        count = len(comprehensive_df[(comprehensive_df['mer'] >= low) & (comprehensive_df['mer'] < high)])
        percentage = count / len(comprehensive_df) * 100
        range_label = f"{low}-{high if high != float('inf') else '100+'}%"
        print(f"   {range_label:8}: {count:4d} samples ({percentage:5.1f}%)")

else:
    print("❌ No samples were successfully processed across all test sets")

print("="*60)

# =========================
# Final Cleanup
# =========================
print("\n Cleaning up...")
torch.cuda.empty_cache()
final_gpu_mem = torch.cuda.memory_allocated(0) // 1024**2
print(f"Final GPU memory: {final_gpu_mem} MB")

gc.collect()
print("✅ Cleanup completed!")
print("🎉 Evaluation finished!")